# EEGNet Model Implementation
Reproduction of EEGNet architecture (Lawhern et al., 2018).
Model structure verified using dummy data without requiring the DEAP dataset.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class EEGNet_Block1(nn.Module):
    def __init__(self):
        super().__init__()
        # Temporal Conv: learns frequency patterns along time axis
        self.temporal_conv = nn.Conv2d(
            in_channels=1,
            out_channels=8,
            kernel_size=(1, 64),
            padding='same',
            bias=False
        )
        # Depthwise Conv: learns spatial patterns along channel axis
        # groups=8 ensures each channel is processed independently
        self.depthwise_conv = nn.Conv2d(
            in_channels=8,
            out_channels=16,
            kernel_size=(32, 1),
            groups=8,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(8)
        self.bn2 = nn.BatchNorm2d(16)
        self.pool = nn.AvgPool2d(kernel_size=(1, 4))
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x):
        x = F.elu(self.bn1(self.temporal_conv(x)))
        x = F.elu(self.bn2(self.depthwise_conv(x)))
        return self.dropout(self.pool(x))

In [ ]:
class EEGNet_Block2(nn.Module):
    def __init__(self):
        super().__init__()
        # Separable Conv = Depthwise + Pointwise
        # Depthwise: summarizes temporal patterns per channel
        self.depthwise_conv = nn.Conv2d(
            in_channels=16,
            out_channels=16,
            kernel_size=(1, 16),
            groups=16,
            padding='same',
            bias=False
        )
        # Pointwise: mixes information across channels
        self.pointwise_conv = nn.Conv2d(
            in_channels=16,
            out_channels=16,
            kernel_size=(1, 1),
            bias=False
        )
        self.bn = nn.BatchNorm2d(16)
        self.pool = nn.AvgPool2d(kernel_size=(1, 8))
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x):
        x = F.elu(self.bn(self.pointwise_conv(self.depthwise_conv(x))))
        return self.dropout(self.pool(x))

In [ ]:
class EEGNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.block1 = EEGNet_Block1()
        self.block2 = EEGNet_Block2()
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.block2(self.block1(x)))

In [ ]:
# Verify model with dummy data (B, 1, 32, 128)
model = EEGNet()
dummy = torch.zeros(32, 1, 32, 128)
out = model(dummy)

print("Output shape:", out.shape)  # (32, 2)
print("Total parameters:", sum(p.numel() for p in model.parameters()))  # 1746